# Osteoporosis MobileNetV2 — Source-Disjoint LOSO CV

## Setup instructions (do this BEFORE running)

1. **Upload the repo as a Kaggle Dataset** named `osteo-repo`:
   - Zip the entire `src/` folder and `scripts/` folder from the repo root.
   - Upload to Kaggle Datasets → New Dataset → name it `osteo-repo`.
   - It will be available at `/kaggle/input/osteo-repo/`.

2. **Upload the 100x100 dataset** as a Kaggle Dataset named `osteo-100x100`:
   - Upload the entire `100x100/` folder (contains train/valid/test).
   - It will be available at `/kaggle/input/osteo-100x100/100x100/`.

3. **Attach both datasets** to this notebook.

4. **Enable GPU** (Accelerator → GPU T4 x2 or P100) and **enable Internet**.

5. **Run All** cells.

6. When complete, download `cv_report.txt` and `cv_report.json` from `/kaggle/working/outputs/`.
   Place them in `outputs/` in your local repo.

## What this notebook does

- Runs 13-fold Leave-One-Source-Out (LOSO) CV, one MobileNetV2 trained per fold.
- **Preprocessing**: channel collapse to grayscale + per-image mean-std standardisation
  (removes brightness confound, fixes 1ch/3ch inconsistency in Osteoporosis).
- **Headline metric**: per-image accuracy via majority vote (one prediction per source X-ray).
- **The prior 97.28% / F1 figures are INVALID (leaked split) and are not reproduced here.**
- Comparison against: (a) 33.3% chance, (b) 53.8% brightness-only LOSO baseline.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
else:
    print('GPU:', torch.cuda.get_device_name(0))

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
assert gpus, 'No GPU visible to TensorFlow — check Kaggle Accelerator setting.'

In [ ]:
# Install ONLY extras not pre-installed on Kaggle.
# Do NOT pin torch/tensorflow — they are preinstalled against Kaggle CUDA.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'opencv-python', 'pyyaml', 'pandas', 'scikit-learn'
])
print('Extras installed.')

In [ ]:
import sys, os
from pathlib import Path

# --- CONFIGURE THESE PATHS ---
# Adjust if your dataset names differ.
REPO_SRC      = '/kaggle/input/osteo-repo/src'          # path to the src/ folder
DATASET_ROOT  = '/kaggle/input/osteo-100x100/100x100'   # path to the 100x100 folder
OUTPUT_DIR    = '/kaggle/working/outputs'
MANIFEST_PATH = '/kaggle/working/manifest.csv'
CV_MODE       = 'LOSO'
EPOCHS        = 20

# Add repo src to Python path so 'import osteo' resolves
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)

# Verify
import osteo
print('osteo imported from:', osteo.__file__)

# Verify dataset
for split in ['train', 'valid', 'test']:
    for cls in ['Normal', 'Osteopenia', 'Osteoporosis']:
        p = Path(DATASET_ROOT) / split / cls
        n = len(list(p.glob('*'))) if p.is_dir() else 0
        print(f'  {split}/{cls}: {n} files')

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from osteo.evaluation.build_manifest import build_manifest
manifest = build_manifest(DATASET_ROOT, MANIFEST_PATH)
print(f'Manifest: {len(manifest):,} rows, {manifest["source_key"].nunique()} sources')

In [ ]:
from osteo.evaluation.cv_runner import run_cv

report_path = run_cv(
    dataset_root=DATASET_ROOT,
    manifest_path=MANIFEST_PATH,
    mode=CV_MODE,
    epochs=EPOCHS,
    output_dir=OUTPUT_DIR,
    keep_weights=False,   # set True to save per-fold .h5 weights (~300 MB each)
)
print(f'\nReport written to: {report_path}')

In [ ]:
# Print the full cv_report.txt
print(Path(report_path).read_text(encoding='utf-8'))

In [ ]:
import os
print('Files in /kaggle/working/outputs:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:50s}  {size_kb:>6} KB')

print('\nDownload cv_report.txt and cv_report.json from the Output panel.')
print('Place them in outputs/ in your local repo.')